# Multimodal Movie Recommender

This notebook trains and compares two rating predictors:

- **Baseline NCF**: learns only from user/movie rating history.
- **Multimodal NCF**: starts from the baseline and adds movie metadata plus poster features as a residual correction.

Read the notebook as a pipeline: load processed data, build feature matrices, split ratings, train the baseline, train the multimodal model, compare behavior, generate recommendations, and save the models.


## Imports

This section checks/imports the packages and sets project constants. The most important switches are `USE_SAMPLE`, `USE_POSTER_SVD`, and the epoch counts.


In [ ]:
import importlib.util
import subprocess
import sys

required_packages = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'matplotlib': 'matplotlib',
    'sklearn': 'scikit-learn',
    'tensorflow': 'tensorflow',
}

missing = [pkg for module, pkg in required_packages.items() if importlib.util.find_spec(module) is None]
if missing:
    print('Installing:', ', '.join(missing))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
else:
    print('All required packages are already installed.')


In [ ]:
import os
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
os.environ.setdefault('TF_ENABLE_ONEDNN_OPTS', '0')

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer, MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.decomposition import TruncatedSVD

print('TensorFlow version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

USE_SAMPLE = True
USE_POSTER_SVD = True
POSTER_SVD_DIM = 64
POSTER_FALLBACK_DIM = 1


def has_training_data(candidate):
    candidate = Path(candidate)
    has_mappings = (candidate / 'movie2idx.csv').exists() and (candidate / 'user2idx.csv').exists()
    has_sample = (candidate / 'model_sample_500k.csv').exists()
    has_full_model = (candidate / 'model_df_clean.csv').exists()
    has_split_full = (candidate / 'final_ratings_clean.csv').exists() and (candidate / 'final_movie_features_clean.csv').exists()
    return has_mappings and (has_sample or has_full_model or has_split_full)


def resolve_data_dir():
    env_dir = os.environ.get('MOVIE_REC_DATA_DIR')
    if env_dir:
        env_path = Path(env_dir).expanduser().resolve()
        if not has_training_data(env_path):
            raise FileNotFoundError(f'MOVIE_REC_DATA_DIR does not contain the processed dataset files: {env_path}')
        return env_path

    candidates = [
        Path.cwd() / 'MultimodalMovieDataset_v2',
        Path.cwd() / 'dataset' / 'MultimodalMovieDataset_v2',
        Path.cwd(),
        Path.cwd() / 'MultimodalMovieDataset',
        Path.cwd() / 'dataset' / 'MultimodalMovieDataset',
    ]
    for candidate in candidates:
        if has_training_data(candidate):
            return candidate.resolve()
    checked = '\n'.join(str(c.resolve()) for c in candidates)
    raise FileNotFoundError('Could not find processed movie recommender data. Checked:\n' + checked)


def resolve_project_root(data_dir):
    data_dir = Path(data_dir).resolve()
    cwd = Path.cwd().resolve()
    if (cwd / 'movie_recommender.ipynb').exists():
        return cwd
    if data_dir.name in {'MultimodalMovieDataset_v2', 'MultimodalMovieDataset'} and data_dir.parent.name == 'dataset':
        return data_dir.parent.parent.resolve()
    return cwd


DATA_DIR = resolve_data_dir()
PROJECT_ROOT = resolve_project_root(DATA_DIR)

EMBEDDING_DIM   = 32
BATCH_SIZE      = 1024
EPOCHS_BASELINE = 20
EPOCHS_MULTI    = 20
SEED            = 42

RATING_MIN = 0.5
RATING_MAX = 5.0

tf.random.set_seed(SEED)
np.random.seed(SEED)

print('Project root:', PROJECT_ROOT)
print('Data folder :', DATA_DIR)


## Load Data

This section loads the processed files created by `dataset/merge.ipynb`:

- rating rows for training/testing,
- user/movie index mappings,
- movie metadata catalog,
- poster embeddings if they already exist.

If `poster_embeddings.npy` is missing, the notebook uses zero poster features so the rest of the pipeline can still be checked. For the real poster experiment, run `extract_poster_embeddings.py` first.


In [ ]:
def require_file(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'Missing required file: {path}')
    return path


TRAINING_USECOLS = [
    'userId', 'movieId', 'rating', 'timestamp', 'user_idx', 'movie_idx',
    'final_title', 'final_genres_str', 'poster_url', 'poster_url_source',
    'poster_imdb_score', 'meta_vote_average', 'meta_vote_count',
    'meta_popularity', 'meta_runtime', 'release_year', 'meta_original_language'
]

MOVIE_CATALOG_USECOLS = [
    'movieId', 'movie_idx', 'final_title', 'final_genres_str', 'poster_url',
    'poster_url_source', 'poster_imdb_score', 'meta_vote_average',
    'meta_vote_count', 'meta_popularity', 'meta_runtime', 'release_year',
    'meta_original_language'
]


def read_training_csv(path):
    return pd.read_csv(path, usecols=lambda c: c in TRAINING_USECOLS)


def load_movie_catalog(data_dir):
    movies_path = data_dir / 'final_movie_features_clean.csv'
    if movies_path.exists():
        return pd.read_csv(movies_path, usecols=lambda c: c in MOVIE_CATALOG_USECOLS)
    return None


def load_training_frame(data_dir, use_sample=True):
    if use_sample:
        sample_path = data_dir / 'model_sample_500k.csv'
        if sample_path.exists():
            print(f'Loading sample file: {sample_path.name}')
            return read_training_csv(sample_path), sample_path.name
        print('model_sample_500k.csv not found; falling back to full data loading.')

    model_path = data_dir / 'model_df_clean.csv'
    if model_path.exists():
        print(f'Loading denormalized full file: {model_path.name}')
        return read_training_csv(model_path), model_path.name

    ratings_path = require_file(data_dir / 'final_ratings_clean.csv')
    movies_path = require_file(data_dir / 'final_movie_features_clean.csv')
    print('model_df_clean.csv not found; merging final_ratings_clean.csv + final_movie_features_clean.csv')
    ratings = pd.read_csv(ratings_path)
    movie_features = pd.read_csv(movies_path, usecols=lambda c: c in MOVIE_CATALOG_USECOLS)
    df_full = ratings.merge(movie_features, on=['movieId', 'movie_idx'], how='left')
    return df_full, 'final_ratings_clean.csv + final_movie_features_clean.csv'


def load_poster_matrix(data_dir, n_movies):
    poster_path = data_dir / 'poster_embeddings.npy'
    if not poster_path.exists():
        print('Warning: poster_embeddings.npy is missing. Using zero poster features so the notebook can still run.')
        print('Run extract_poster_embeddings.py later to enable the visual poster branch.')
        poster_raw = np.zeros((n_movies, POSTER_FALLBACK_DIM), dtype=np.float32)
        return poster_raw, False

    poster_raw = np.load(poster_path).astype(np.float32, copy=False)
    if poster_raw.ndim != 2:
        raise ValueError(f'poster_embeddings.npy must be 2-D, got shape {poster_raw.shape}')

    if poster_raw.shape[0] < n_movies:
        pad = np.zeros((n_movies - poster_raw.shape[0], poster_raw.shape[1]), dtype=np.float32)
        poster_raw = np.vstack([poster_raw, pad])
        print(f'Padded poster embeddings to {poster_raw.shape} to match movie2idx.csv')
    elif poster_raw.shape[0] > n_movies:
        poster_raw = poster_raw[:n_movies]
        print(f'Trimmed poster embeddings to {poster_raw.shape} to match movie2idx.csv')
    return poster_raw, True


df, loaded_from = load_training_frame(DATA_DIR, USE_SAMPLE)
movie_catalog = load_movie_catalog(DATA_DIR)
if movie_catalog is None:
    movie_catalog = df[MOVIE_CATALOG_USECOLS].drop_duplicates('movie_idx').copy()

user2idx_df  = pd.read_csv(require_file(DATA_DIR / 'user2idx.csv'))
movie2idx_df = pd.read_csv(require_file(DATA_DIR / 'movie2idx.csv'))

N_USERS  = int(user2idx_df['user_idx'].max()) + 1
N_MOVIES = int(movie2idx_df['movie_idx'].max()) + 1

if df['user_idx'].max() >= N_USERS:
    raise ValueError('df contains user_idx values outside user2idx.csv')
if df['movie_idx'].max() >= N_MOVIES:
    raise ValueError('df contains movie_idx values outside movie2idx.csv')

poster_raw, poster_file_found = load_poster_matrix(DATA_DIR, N_MOVIES)
poster_available = np.any(poster_raw != 0, axis=1)
unique_df_movies = np.sort(df['movie_idx'].unique())
covered_in_df = int(poster_available[unique_df_movies].sum())

poster_model = np.zeros_like(poster_raw, dtype=np.float32)
if poster_available.any():
    poster_mean = poster_raw[poster_available].mean(axis=0, keepdims=True)
    poster_std = poster_raw[poster_available].std(axis=0, keepdims=True) + 1e-6
    poster_model[poster_available] = (poster_raw[poster_available] - poster_mean) / poster_std

if USE_POSTER_SVD and poster_model.shape[1] > POSTER_SVD_DIM and poster_available.sum() > POSTER_SVD_DIM:
    svd = TruncatedSVD(n_components=POSTER_SVD_DIM, random_state=SEED)
    reduced = np.zeros((N_MOVIES, POSTER_SVD_DIM), dtype=np.float32)
    reduced_present = svd.fit_transform(poster_model[poster_available]).astype(np.float32)
    red_mean = reduced_present.mean(axis=0, keepdims=True)
    red_std = reduced_present.std(axis=0, keepdims=True) + 1e-6
    reduced[poster_available] = (reduced_present - red_mean) / red_std
    poster_model = reduced
    print(f'Poster SVD dim: {POSTER_SVD_DIM}, explained variance: {svd.explained_variance_ratio_.sum():.3f}')

poster_embeddings = np.concatenate(
    [poster_model.astype(np.float32), poster_available.astype(np.float32)[:, None]],
    axis=1
)
POSTER_DIM = poster_embeddings.shape[1]

del poster_raw

print(f'Rows loaded       : {len(df):,} from {loaded_from}')
print(f'Total users       : {N_USERS:,}')
print(f'Total movies      : {N_MOVIES:,}')
print(f'Movie catalog     : {len(movie_catalog):,} rows')
print(f'Poster file found : {poster_file_found}')
print(f'Poster dim        : {POSTER_DIM} ({POSTER_DIM - 1} visual + availability flag)')
print(f'Poster coverage   : {poster_available.sum():,} / {N_MOVIES:,} movies')
print(f'Coverage in df     : {covered_in_df:,} / {len(unique_df_movies):,} unique movies')
if 'poster_url' in df.columns:
    rows_with_poster_url = int(df['poster_url'].notna().sum())
    unique_with_poster_url = int(df.drop_duplicates('movie_idx')['poster_url'].notna().sum())
    print(f'Rows with poster URL in df: {rows_with_poster_url:,} / {len(df):,}')
    print(f'Unique movies with poster URL in df: {unique_with_poster_url:,} / {len(unique_df_movies):,}')
    if poster_file_found and unique_with_poster_url - covered_in_df > 500:
        print('Warning: many movies have poster URLs but no embedding yet. Re-run extract_poster_embeddings.py for this DATA_DIR.')
print(f'Rating range      : {df.rating.min()} - {df.rating.max()}')
df.head(3)


## Data Check

Before training, this section checks the shape of the sample: rating distribution, activity per user/movie, and missing values. These plots help explain whether the model is learning from dense or sparse behavior.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

df['rating'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Rating Distribution')
axes[0].set_xlabel('Rating')

ratings_per_user = df.groupby('user_idx').size()
axes[1].hist(ratings_per_user, bins=50, color='coral', log=True)
axes[1].set_title('Ratings per User')
axes[1].set_xlabel('Count')

ratings_per_movie = df.groupby('movie_idx').size()
axes[2].hist(ratings_per_movie, bins=50, color='green', log=True)
axes[2].set_title('Ratings per Movie')
axes[2].set_xlabel('Count')

plt.tight_layout()
plt.show()

print('Missing values:')
print(df[['user_idx','movie_idx','rating','final_genres_str',
          'meta_vote_average','meta_vote_count','meta_popularity',
          'meta_runtime','release_year']].isnull().sum())


## Metadata Features

The multimodal model needs fixed-length side features for each movie. Here, genres become one-hot columns and numeric TMDB fields become scaled numeric features.


In [ ]:
movie_features = df[['movie_idx','final_genres_str','meta_vote_average',
                      'meta_vote_count','meta_popularity',
                      'meta_runtime','release_year']].drop_duplicates('movie_idx').copy()

movie_features['final_genres_str'] = movie_features['final_genres_str'].fillna('')
genre_lists = movie_features['final_genres_str'].apply(lambda x: x.split('|') if x else ['Unknown'])

mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(genre_lists)

print(f'Genres found ({len(mlb.classes_)}):', list(mlb.classes_))
print(f'Genre matrix shape: {genre_matrix.shape}')


In [ ]:
numeric_cols = ['meta_vote_average', 'meta_vote_count', 'meta_popularity',
                'meta_runtime', 'release_year']

for col in numeric_cols:
    movie_features[col] = pd.to_numeric(movie_features[col], errors='coerce')

for col in ['meta_vote_count', 'meta_popularity']:
    movie_features[col] = np.log1p(movie_features[col].clip(lower=0))

for col in numeric_cols:
    median = movie_features[col].median()
    fill_value = 0.0 if pd.isna(median) else median
    movie_features[col] = movie_features[col].fillna(fill_value)

scaler = MinMaxScaler()
numeric_matrix = scaler.fit_transform(movie_features[numeric_cols])

metadata_matrix = np.hstack([genre_matrix, numeric_matrix]).astype(np.float32)
metadata_feature_names = list(mlb.classes_) + numeric_cols
metadata_feature_groups = [(name, [idx]) for idx, name in enumerate(metadata_feature_names)]
META_DIM = metadata_matrix.shape[1]

print(f'Metadata vector dimension: {META_DIM}  (genres={genre_matrix.shape[1]}, numerics={numeric_matrix.shape[1]})')
print('Numeric metadata columns:', numeric_cols)

meta_idx_array = movie_features['movie_idx'].values
meta_lookup = np.zeros((N_MOVIES, META_DIM), dtype=np.float32)
meta_lookup[meta_idx_array] = metadata_matrix

print(f'meta_lookup shape: {meta_lookup.shape}')


## Split

The loaded rating rows are split into train/validation/test sets: 80% for fitting, 10% for early stopping and residual tuning, and 10% for final reporting.


In [ ]:
user_arr   = df['user_idx'].values.astype(np.int32)
movie_arr  = df['movie_idx'].values.astype(np.int32)
rating_arr = df['rating'].values.astype(np.float32)

idx = np.arange(len(df))
train_idx, temp_idx = train_test_split(idx, test_size=0.2, random_state=SEED)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=SEED)

print(f'Train size : {len(train_idx):,}')
print(f'Val size   : {len(val_idx):,}')
print(f'Test size  : {len(test_idx):,}')


def make_inputs(idx_set):
    u = user_arr[idx_set]
    m = movie_arr[idx_set]
    r = rating_arr[idx_set]
    meta = meta_lookup[m]
    poster = poster_embeddings[m]
    return u, m, r, meta, poster


u_train, m_train, r_train, meta_train, poster_train = make_inputs(train_idx)
u_val,   m_val,   r_val,   meta_val,   poster_val   = make_inputs(val_idx)
u_test,  m_test,  r_test,  meta_test,  poster_test  = make_inputs(test_idx)

print('\nInput shapes (train):')
print(f'  users  : {u_train.shape}')
print(f'  movies : {m_train.shape}')
print(f'  ratings: {r_train.shape}')
print(f'  meta   : {meta_train.shape}')
print(f'  poster : {poster_train.shape}')


## Evaluation

RMSE and MAE are the main metrics. RMSE penalizes large mistakes more strongly, while MAE is easier to read as average rating error.


In [ ]:
def predict_ratings(model, inputs_dict, batch_size=BATCH_SIZE):
    y_pred = model.predict(inputs_dict, batch_size=batch_size, verbose=0).flatten()
    return np.clip(y_pred, RATING_MIN, RATING_MAX)


def metric_dict(y_true, y_pred):
    return {
        'RMSE': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'MAE': float(mean_absolute_error(y_true, y_pred)),
    }


def evaluate_model(model, inputs_dict, y_true, model_name):
    y_pred = predict_ratings(model, inputs_dict)
    metrics = metric_dict(y_true, y_pred)
    print(f'[{model_name}]  RMSE = {metrics["RMSE"]:.4f}  |  MAE = {metrics["MAE"]:.4f}')
    return metrics['RMSE'], metrics['MAE'], y_pred


results = {}
test_predictions = {}


## Baseline Model

The baseline neural collaborative filtering model learns a vector for each user and movie, concatenates them, and predicts the rating from interaction history only. This is the reference model.


In [ ]:
def build_baseline_model(n_users, n_movies, embed_dim=EMBEDDING_DIM):
    user_input  = keras.Input(shape=(1,), name='user_idx',  dtype='int32')
    movie_input = keras.Input(shape=(1,), name='movie_idx', dtype='int32')

    user_emb = layers.Embedding(
        input_dim=n_users,
        output_dim=embed_dim,
        embeddings_regularizer=keras.regularizers.l2(1e-6),
        name='user_embedding'
    )(user_input)
    user_emb = layers.Flatten(name='user_flatten')(user_emb)

    movie_emb = layers.Embedding(
        input_dim=n_movies,
        output_dim=embed_dim,
        embeddings_regularizer=keras.regularizers.l2(1e-6),
        name='movie_embedding'
    )(movie_input)
    movie_emb = layers.Flatten(name='movie_flatten')(movie_emb)

    x = layers.Concatenate(name='cf_features')([user_emb, movie_emb])
    x = layers.Dense(64, activation='relu', name='cf_dense_1')(x)
    x = layers.Dropout(0.3, name='cf_dropout_1')(x)
    x = layers.Dense(32, activation='relu', name='cf_dense_2')(x)
    output = layers.Dense(1, name='rating_output')(x)

    model = Model(inputs=[user_input, movie_input], outputs=output, name='Baseline_NCF')
    return model


baseline_model = build_baseline_model(N_USERS, N_MOVIES)
baseline_model.summary()

baseline_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='mse',
    metrics=['mae']
)


In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=3, restore_best_weights=True
)

history_baseline = baseline_model.fit(
    x={'user_idx': u_train, 'movie_idx': m_train},
    y=r_train,
    validation_data=(
        {'user_idx': u_val, 'movie_idx': m_val},
        r_val
    ),
    batch_size=BATCH_SIZE,
    epochs=EPOCHS_BASELINE,
    callbacks=[early_stop],
    verbose=1
)


In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history_baseline.history['loss'],     label='train loss')
plt.plot(history_baseline.history['val_loss'], label='val loss')
plt.title('Baseline Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_baseline.history['mae'],     label='train MAE')
plt.plot(history_baseline.history['val_mae'], label='val MAE')
plt.title('Baseline MAE')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
rmse_b, mae_b, baseline_test_pred = evaluate_model(
    baseline_model,
    {'user_idx': u_test, 'movie_idx': m_test},
    r_test,
    model_name='Baseline NCF'
)
results['Baseline NCF'] = {'RMSE': rmse_b, 'MAE': mae_b}
test_predictions['Baseline NCF'] = baseline_test_pred


## Multimodal Model

The multimodal model copies the trained baseline weights, freezes that collaborative branch first, and trains a small residual branch from movie metadata and poster features. After that, it keeps the best residual strength on validation data.


### Diagnostics

These checks show how much poster signal is actually available and whether poster features have any simple relationship with average movie ratings.


In [ ]:
zero_poster_mask = ~poster_available
print(f'Movies with all-zero raw poster embeddings: {zero_poster_mask.sum()} / {len(poster_available)}')
print(f'Poster model matrix: shape={poster_embeddings.shape}, mean={poster_embeddings[:, :-1].mean():.3f}, std={poster_embeddings[:, :-1].std():.3f}')

rv = df.groupby('movie_idx')['rating'].agg(['var', 'count'])
print('\nPer-movie rating variance:')
print(f'  Mean   : {rv["var"].mean():.4f}')
print(f'  Median : {rv["var"].median():.4f}')
print(f'  Movies with variance > 1.0 : {(rv["var"] > 1.0).sum()} / {len(rv)}')

sample_idxs = df['movie_idx'].unique()
available_sample = sample_idxs[poster_available[sample_idxs]]
if len(available_sample) > 2 and poster_embeddings.shape[1] > 2:
    poster_signal = poster_embeddings[available_sample, :-1]
    svd_diag = TruncatedSVD(n_components=1, random_state=SEED)
    poster_pc1 = svd_diag.fit_transform(poster_signal).flatten()
    avg_rating = df.groupby('movie_idx')['rating'].mean().loc[available_sample].values
    corr = np.corrcoef(poster_pc1, avg_rating)[0, 1]
    print(f'\nPearson corr(poster PC1, avg movie rating) = {corr:.4f}')
else:
    print('\nNot enough available posters for PC1 correlation diagnostic.')

print(f'\nBaseline trainable params: {baseline_model.count_params():,}')


In [ ]:
def build_multimodal_model(n_users, n_movies, meta_dim, poster_dim, embed_dim=EMBEDDING_DIM):
    user_input   = keras.Input(shape=(1,),          name='user_idx',   dtype='int32')
    movie_input  = keras.Input(shape=(1,),          name='movie_idx',  dtype='int32')
    meta_input   = keras.Input(shape=(meta_dim,),   name='metadata')
    poster_input = keras.Input(shape=(poster_dim,), name='poster_emb')

    user_emb = layers.Embedding(
        n_users, embed_dim,
        embeddings_regularizer=keras.regularizers.l2(1e-6),
        name='user_embedding'
    )(user_input)
    user_emb = layers.Flatten(name='user_flatten')(user_emb)

    movie_emb = layers.Embedding(
        n_movies, embed_dim,
        embeddings_regularizer=keras.regularizers.l2(1e-6),
        name='movie_embedding'
    )(movie_input)
    movie_emb = layers.Flatten(name='movie_flatten')(movie_emb)

    cf = layers.Concatenate(name='cf_features')([user_emb, movie_emb])
    cf = layers.Dense(64, activation='relu', name='cf_dense_1')(cf)
    cf = layers.Dense(32, activation='relu', name='cf_dense_2')(cf)
    cf_rating = layers.Dense(1, name='cf_rating')(cf)

    meta = layers.BatchNormalization(name='meta_bn')(meta_input)
    meta = layers.Dense(
        16,
        activation='relu',
        kernel_regularizer=keras.regularizers.l2(1e-4),
        name='meta_dense_1'
    )(meta)
    meta = layers.Dropout(0.15, name='meta_dropout_1')(meta)

    poster = layers.BatchNormalization(name='poster_bn')(poster_input)
    poster = layers.Dense(
        32,
        activation='relu',
        kernel_regularizer=keras.regularizers.l2(1e-4),
        name='poster_dense_1'
    )(poster)
    poster = layers.Dropout(0.25, name='poster_dropout_1')(poster)

    side = layers.Concatenate(name='side_features')([meta, poster])
    side = layers.Dense(
        32,
        activation='relu',
        kernel_regularizer=keras.regularizers.l2(1e-4),
        name='side_dense_1'
    )(side)
    side = layers.Dropout(0.20, name='side_dropout_1')(side)
    residual = layers.Dense(
        1,
        kernel_initializer='zeros',
        bias_initializer='zeros',
        name='side_residual'
    )(side)

    output = layers.Add(name='rating_output')([cf_rating, residual])
    model = Model(
        inputs=[user_input, movie_input, meta_input, poster_input],
        outputs=output,
        name='Multimodal_NCF_residual'
    )
    return model


def copy_baseline_to_multimodal(baseline, multimodal):
    layer_pairs = {
        'user_embedding': 'user_embedding',
        'movie_embedding': 'movie_embedding',
        'cf_dense_1': 'cf_dense_1',
        'cf_dense_2': 'cf_dense_2',
        'rating_output': 'cf_rating',
    }
    for src_name, dst_name in layer_pairs.items():
        multimodal.get_layer(dst_name).set_weights(baseline.get_layer(src_name).get_weights())


def set_collaborative_trainable(model, trainable):
    for name in ['user_embedding', 'movie_embedding', 'cf_dense_1', 'cf_dense_2', 'cf_rating']:
        model.get_layer(name).trainable = trainable


multi_model = build_multimodal_model(N_USERS, N_MOVIES, META_DIM, POSTER_DIM)
copy_baseline_to_multimodal(baseline_model, multi_model)
set_collaborative_trainable(multi_model, False)

multi_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3, clipnorm=1.0),
    loss='mse',
    metrics=['mae']
)

multi_model.summary()
print(f'Multimodal params: total={multi_model.count_params():,}, trainable side branch only')


In [ ]:
multi_train_inputs = {
    'user_idx'  : u_train,
    'movie_idx' : m_train,
    'metadata'  : meta_train,
    'poster_emb': poster_train,
}
multi_val_inputs = {
    'user_idx'  : u_val,
    'movie_idx' : m_val,
    'metadata'  : meta_val,
    'poster_emb': poster_val,
}

initial_weights = multi_model.get_weights()
initial_val_pred = multi_model.predict(multi_val_inputs, batch_size=BATCH_SIZE, verbose=0).flatten()
initial_val_pred = np.clip(initial_val_pred, RATING_MIN, RATING_MAX)
initial_val_rmse = np.sqrt(mean_squared_error(r_val, initial_val_pred))
print(f'Initial multimodal validation RMSE (baseline-equivalent): {initial_val_rmse:.4f}')

early_stop_multi = keras.callbacks.EarlyStopping(
    monitor='val_mae', mode='min', patience=4, restore_best_weights=True
)
reduce_lr_multi = keras.callbacks.ReduceLROnPlateau(
    monitor='val_mae', mode='min', factor=0.5, patience=2, min_lr=1e-6, verbose=1
)

history_multi = multi_model.fit(
    x=multi_train_inputs,
    y=r_train,
    validation_data=(multi_val_inputs, r_val),
    batch_size=BATCH_SIZE,
    epochs=EPOCHS_MULTI,
    callbacks=[early_stop_multi, reduce_lr_multi],
    verbose=1
)

trained_weights = multi_model.get_weights()
alpha_grid = np.linspace(0.0, 1.0, 21)
best_alpha = 0.0
best_val_rmse = initial_val_rmse
best_weights = initial_weights

for alpha in alpha_grid[1:]:
    multi_model.set_weights(trained_weights)
    residual_layer = multi_model.get_layer('side_residual')
    residual_weights = residual_layer.get_weights()
    residual_layer.set_weights([residual_weights[0] * alpha, residual_weights[1] * alpha])

    val_pred = multi_model.predict(multi_val_inputs, batch_size=BATCH_SIZE, verbose=0).flatten()
    val_pred = np.clip(val_pred, RATING_MIN, RATING_MAX)
    val_rmse = np.sqrt(mean_squared_error(r_val, val_pred))
    if val_rmse < best_val_rmse - 1e-5:
        best_alpha = float(alpha)
        best_val_rmse = float(val_rmse)
        best_weights = multi_model.get_weights()

multi_model.set_weights(best_weights)
if best_alpha == 0.0:
    print('Side residual did not improve validation RMSE; restored baseline-equivalent multimodal weights.')
else:
    print(f'Best residual alpha on validation: {best_alpha:.2f} (RMSE {best_val_rmse:.4f})')


In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history_multi.history['loss'],     label='train loss')
plt.plot(history_multi.history['val_loss'], label='val loss')
plt.title('Multimodal Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_multi.history['mae'],     label='train MAE')
plt.plot(history_multi.history['val_mae'], label='val MAE')
plt.title('Multimodal MAE')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
multi_test_inputs = {
    'user_idx'  : u_test,
    'movie_idx' : m_test,
    'metadata'  : meta_test,
    'poster_emb': poster_test
}

rmse_m, mae_m, multi_test_pred = evaluate_model(
    multi_model,
    multi_test_inputs,
    r_test,
    model_name='Multimodal NCF'
)
results['Multimodal NCF'] = {'RMSE': rmse_m, 'MAE': mae_m}
test_predictions['Multimodal NCF'] = multi_test_pred


## Comparison

This section compares the two models on the held-out test set and prints the architecture differences that are useful for slides.


In [ ]:
comparison_df = pd.DataFrame(results).T
comparison_df['RMSE_improvement_%'] = (
    (comparison_df.loc['Baseline NCF', 'RMSE'] - comparison_df['RMSE'])
    / comparison_df.loc['Baseline NCF', 'RMSE'] * 100
).round(2)
comparison_df['MAE_improvement_%'] = (
    (comparison_df.loc['Baseline NCF', 'MAE'] - comparison_df['MAE'])
    / comparison_df.loc['Baseline NCF', 'MAE'] * 100
).round(2)

print('\n=== Model Comparison (Test Set) ===')
print(comparison_df.to_string(float_format='{:.4f}'.format))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
comparison_df[['RMSE']].plot(kind='bar', ax=axes[0], color=['steelblue', 'coral'], legend=False)
axes[0].set_title('RMSE')
axes[0].set_ylabel('Lower is better')
axes[0].set_xticklabels(comparison_df.index, rotation=15, ha='right')

comparison_df[['MAE']].plot(kind='bar', ax=axes[1], color=['steelblue', 'coral'], legend=False)
axes[1].set_title('MAE')
axes[1].set_ylabel('Lower is better')
axes[1].set_xticklabels(comparison_df.index, rotation=15, ha='right')

improvement_cols = ['RMSE_improvement_%', 'MAE_improvement_%']
comparison_df.loc[['Multimodal NCF'], improvement_cols].T.plot(kind='bar', ax=axes[2], color='seagreen', legend=False)
axes[2].axhline(0, color='black', linewidth=0.8)
axes[2].set_title('Multimodal Improvement %')
axes[2].set_ylabel('% vs baseline')
axes[2].set_xticklabels(['RMSE', 'MAE'], rotation=0)
plt.tight_layout()
plt.show()


def trainable_param_count(model):
    return int(np.sum([np.prod(v.shape) for v in model.trainable_weights]))

arch_df = pd.DataFrame({
    'Model': ['Baseline NCF', 'Multimodal NCF'],
    'User Embedding': [f'Yes ({EMBEDDING_DIM}-dim)', f'Yes ({EMBEDDING_DIM}-dim, warm-started)'],
    'Movie Embedding': [f'Yes ({EMBEDDING_DIM}-dim)', f'Yes ({EMBEDDING_DIM}-dim, warm-started)'],
    'Metadata': ['No', f'Yes ({META_DIM}-dim residual)'],
    'Poster CNN': ['No', f'Yes ({POSTER_DIM - 1} visual + availability flag)'],
    'Total Params': [f'{baseline_model.count_params():,}', f'{multi_model.count_params():,}'],
    'Trainable Params': [f'{trainable_param_count(baseline_model):,}', f'{trainable_param_count(multi_model):,}'],
}).set_index('Model')

print('\n=== Architecture Comparison ===')
print(arch_df.to_string())


## Model Behavior Visualizations

These plots explain how the models behave, not only which one has the lower average error. They show where the multimodal branch changes predictions and which side features matter most.


In [ ]:
eval_df = pd.DataFrame({
    'user_idx': u_test,
    'movie_idx': m_test,
    'actual': r_test,
    'baseline_pred': test_predictions['Baseline NCF'],
    'multimodal_pred': test_predictions['Multimodal NCF'],
})
eval_df['baseline_error'] = eval_df['baseline_pred'] - eval_df['actual']
eval_df['multimodal_error'] = eval_df['multimodal_pred'] - eval_df['actual']
eval_df['baseline_abs_error'] = eval_df['baseline_error'].abs()
eval_df['multimodal_abs_error'] = eval_df['multimodal_error'].abs()
eval_df['abs_error_delta'] = eval_df['baseline_abs_error'] - eval_df['multimodal_abs_error']
eval_df['model_prediction_delta'] = eval_df['multimodal_pred'] - eval_df['baseline_pred']
eval_df['poster_available'] = poster_available[eval_df['movie_idx'].values]

user_activity = df.groupby('user_idx').size()
movie_activity = df.groupby('movie_idx').size()
eval_df['user_activity'] = eval_df['user_idx'].map(user_activity).fillna(0).astype(int)
eval_df['movie_activity'] = eval_df['movie_idx'].map(movie_activity).fillna(0).astype(int)
rating_categories = sorted(pd.Series(eval_df['actual']).unique())
eval_df['rating_bucket'] = pd.Categorical(eval_df['actual'], categories=rating_categories, ordered=True)

def activity_bins(values, labels):
    if len(values) < len(labels):
        return pd.Series(['all'] * len(values), index=values.index)
    try:
        return pd.qcut(values.rank(method='first'), q=len(labels), labels=labels)
    except ValueError:
        return pd.Series(['all'] * len(values), index=values.index)

eval_df['user_activity_bin'] = activity_bins(eval_df['user_activity'], ['low', 'mid-low', 'mid-high', 'high'])
eval_df['movie_activity_bin'] = activity_bins(eval_df['movie_activity'], ['niche', 'mid-niche', 'mid-popular', 'popular'])

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[0, 0].hist(eval_df['baseline_error'], bins=60, alpha=0.65, label='Baseline', color='steelblue')
axes[0, 0].hist(eval_df['multimodal_error'], bins=60, alpha=0.65, label='Multimodal', color='coral')
axes[0, 0].axvline(0, color='black', linewidth=0.8)
axes[0, 0].set_title('Prediction Error Distribution')
axes[0, 0].set_xlabel('predicted - actual rating')
axes[0, 0].legend()

sample_plot = eval_df.sample(n=min(12000, len(eval_df)), random_state=SEED)
axes[0, 1].scatter(sample_plot['actual'], sample_plot['baseline_pred'], s=8, alpha=0.25, label='Baseline', color='steelblue')
axes[0, 1].scatter(sample_plot['actual'], sample_plot['multimodal_pred'], s=8, alpha=0.25, label='Multimodal', color='coral')
axes[0, 1].plot([RATING_MIN, RATING_MAX], [RATING_MIN, RATING_MAX], color='black', linewidth=1)
axes[0, 1].set_title('Actual vs Predicted Ratings')
axes[0, 1].set_xlabel('Actual')
axes[0, 1].set_ylabel('Predicted')
axes[0, 1].legend()

bucket_mae = eval_df.groupby('rating_bucket', observed=True)[['baseline_abs_error', 'multimodal_abs_error']].mean()
bucket_mae.rename(columns={'baseline_abs_error': 'Baseline', 'multimodal_abs_error': 'Multimodal'}).plot(
    kind='bar', ax=axes[1, 0], color=['steelblue', 'coral']
)
axes[1, 0].set_title('MAE by True Rating')
axes[1, 0].set_xlabel('Actual rating')
axes[1, 0].set_ylabel('MAE')
axes[1, 0].tick_params(axis='x', rotation=0)

axes[1, 1].hist(eval_df['model_prediction_delta'], bins=60, color='seagreen', alpha=0.75)
axes[1, 1].axvline(0, color='black', linewidth=0.8)
axes[1, 1].set_title('How Much Multimodal Changes Predictions')
axes[1, 1].set_xlabel('multimodal prediction - baseline prediction')
plt.tight_layout()
plt.show()

print('Positive abs_error_delta means multimodal reduced absolute error.')
print(eval_df[['baseline_abs_error', 'multimodal_abs_error', 'abs_error_delta', 'model_prediction_delta']].describe().round(4))


In [ ]:
def rmse_np(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def summarize_segment(frame, segment_col):
    rows = []
    for segment_value, group in frame.groupby(segment_col, observed=True):
        if len(group) < 30:
            continue
        rows.append({
            'segment': str(segment_value),
            'n': len(group),
            'baseline_RMSE': rmse_np(group['actual'], group['baseline_pred']),
            'multimodal_RMSE': rmse_np(group['actual'], group['multimodal_pred']),
            'baseline_MAE': float(group['baseline_abs_error'].mean()),
            'multimodal_MAE': float(group['multimodal_abs_error'].mean()),
        })
    out = pd.DataFrame(rows)
    if out.empty:
        return out
    out['RMSE_gain_%'] = (out['baseline_RMSE'] - out['multimodal_RMSE']) / out['baseline_RMSE'] * 100
    out['MAE_gain_%'] = (out['baseline_MAE'] - out['multimodal_MAE']) / out['baseline_MAE'] * 100
    return out

segments = {
    'Poster available': 'poster_available',
    'Movie activity': 'movie_activity_bin',
    'User activity': 'user_activity_bin',
}

fig, axes = plt.subplots(1, len(segments), figsize=(16, 4), sharey=False)
segment_tables = {}
for ax, (title, column) in zip(axes, segments.items()):
    table = summarize_segment(eval_df, column)
    segment_tables[title] = table
    if table.empty:
        ax.set_title(title)
        ax.text(0.5, 0.5, 'Not enough data', ha='center', va='center')
        continue
    plot_df = table.set_index('segment')[['baseline_RMSE', 'multimodal_RMSE']]
    plot_df.plot(kind='bar', ax=ax, color=['steelblue', 'coral'])
    ax.set_title(title)
    ax.set_ylabel('RMSE')
    ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

for title, table in segment_tables.items():
    print(f'\n=== {title} ===')
    if table.empty:
        print('Not enough data')
    else:
        print(table.to_string(index=False, float_format='{:.4f}'.format))


In [ ]:
def permutation_importance_multimodal(model, base_inputs, y_true, max_rows=10000, seed=SEED):
    rng = np.random.default_rng(seed)
    n = min(max_rows, len(y_true))
    sample_idx = rng.choice(len(y_true), size=n, replace=False)
    y_sample = y_true[sample_idx]
    sampled_inputs = {name: values[sample_idx].copy() for name, values in base_inputs.items()}

    base_pred = predict_ratings(model, sampled_inputs)
    base_rmse = rmse_np(y_sample, base_pred)
    base_mae = float(mean_absolute_error(y_sample, base_pred))

    groups = [(name, 'metadata', cols) for name, cols in metadata_feature_groups]
    poster_visual = sampled_inputs['poster_emb'][:, :-1]
    poster_flag = sampled_inputs['poster_emb'][:, -1]
    if poster_visual.shape[1] > 0 and np.any(poster_visual != 0):
        groups.append(('poster_visual_embedding', 'poster_visual', None))
    if np.unique(poster_flag).size > 1:
        groups.append(('poster_available_flag', 'poster_flag', None))

    rows = []
    for name, source, cols in groups:
        perturbed = {key: value.copy() for key, value in sampled_inputs.items()}
        if source == 'metadata':
            for col in cols:
                perturbed['metadata'][:, col] = rng.permutation(perturbed['metadata'][:, col])
        elif source == 'poster_visual':
            perm = rng.permutation(n)
            perturbed['poster_emb'][:, :-1] = perturbed['poster_emb'][perm, :-1]
        elif source == 'poster_flag':
            perturbed['poster_emb'][:, -1] = rng.permutation(perturbed['poster_emb'][:, -1])

        pred = predict_ratings(model, perturbed)
        rmse = rmse_np(y_sample, pred)
        mae = float(mean_absolute_error(y_sample, pred))
        rows.append({
            'feature_group': name,
            'RMSE_when_permuted': rmse,
            'MAE_when_permuted': mae,
            'RMSE_increase': rmse - base_rmse,
            'MAE_increase': mae - base_mae,
        })

    return pd.DataFrame(rows).sort_values('RMSE_increase', ascending=False).reset_index(drop=True), base_rmse, base_mae

feature_importance_df, base_perm_rmse, base_perm_mae = permutation_importance_multimodal(
    multi_model,
    multi_test_inputs,
    r_test,
)

print(f'Base multimodal subset RMSE={base_perm_rmse:.4f}, MAE={base_perm_mae:.4f}')
print(feature_importance_df.head(15).to_string(index=False, float_format='{:.5f}'.format))

top_importance = feature_importance_df.head(15).iloc[::-1]
plt.figure(figsize=(10, 6))
plt.barh(top_importance['feature_group'], top_importance['RMSE_increase'], color='seagreen')
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Multimodal Side-Feature Importance')
plt.xlabel('RMSE increase after permutation')
plt.tight_layout()
plt.show()


## Recommendations

This demo recommends movies for an active training user. It only ranks movies whose embeddings were actually trained, which avoids sample-mode recommendations from random untrained movie vectors.


In [ ]:
def build_recommendation_catalog(movie_catalog, df_train, min_movie_ratings=5):
    rating_counts = df_train.groupby('movie_idx').size().rename('rating_count')
    catalog = movie_catalog.drop_duplicates('movie_idx').set_index('movie_idx').copy()
    catalog = catalog.join(rating_counts, how='inner')
    trained_movie_idxs = np.unique(m_train)
    catalog = catalog.loc[catalog.index.intersection(trained_movie_idxs)]

    filtered = catalog[catalog['rating_count'] >= min_movie_ratings]
    if filtered.empty:
        print(f'No movies met min_movie_ratings={min_movie_ratings}; falling back to all trained candidate movies.')
        filtered = catalog
    if filtered.empty:
        raise ValueError('No trained movie candidates are available for recommendation.')
    return filtered.sort_values('rating_count', ascending=False)


def choose_demo_user(min_train_ratings=10):
    train_user_counts = pd.Series(u_train).value_counts()
    eligible = train_user_counts[train_user_counts >= min_train_ratings]
    if eligible.empty:
        return int(train_user_counts.index[0])
    return int(eligible.index[0])


def recommend_top_n(model, user_id, df_all, poster_emb_arr, meta_lookup_arr,
                    movie_catalog_df, n=10, use_multimodal=False):
    seen_movies = set(df_all[df_all['user_idx'] == user_id]['movie_idx'].values)
    candidate_movies = movie_catalog_df.index.to_numpy(dtype=np.int32)
    if seen_movies:
        candidate_movies = candidate_movies[~np.isin(candidate_movies, list(seen_movies))]
    if len(candidate_movies) == 0:
        raise ValueError(f'No recommendation candidates left for user_idx={user_id}.')

    users_arr = np.full(len(candidate_movies), user_id, dtype=np.int32)
    movies_arr = candidate_movies.astype(np.int32)

    if use_multimodal:
        inputs = {
            'user_idx'  : users_arr,
            'movie_idx' : movies_arr,
            'metadata'  : meta_lookup_arr[movies_arr],
            'poster_emb': poster_emb_arr[movies_arr]
        }
    else:
        inputs = {'user_idx': users_arr, 'movie_idx': movies_arr}

    preds = predict_ratings(model, inputs, batch_size=2048)
    top_n_idx = np.argsort(preds)[::-1][:n]
    top_movies = movies_arr[top_n_idx]
    top_scores = preds[top_n_idx]

    recs = []
    for rank, (m_idx, score) in enumerate(zip(top_movies, top_scores), start=1):
        row = movie_catalog_df.loc[m_idx]
        recs.append({
            'rank': rank,
            'movie_idx': int(m_idx),
            'title': row.get('final_title', f'movie_{m_idx}'),
            'genres': row.get('final_genres_str', ''),
            'poster_source': row.get('poster_url_source', ''),
            'rating_count': int(row.get('rating_count', 0)),
            'predicted_rating': round(float(score), 3),
        })
    return pd.DataFrame(recs)


recommendation_catalog = build_recommendation_catalog(movie_catalog, df, min_movie_ratings=5)
DEMO_USER = choose_demo_user(min_train_ratings=10)
print(f'Recommendation candidate movies: {len(recommendation_catalog):,}')
print(f'Top-10 recommendations for user_idx={DEMO_USER}')

baseline_recs = recommend_top_n(
    baseline_model, DEMO_USER, df, poster_embeddings, meta_lookup,
    recommendation_catalog, n=10, use_multimodal=False
)
multimodal_recs = recommend_top_n(
    multi_model, DEMO_USER, df, poster_embeddings, meta_lookup,
    recommendation_catalog, n=10, use_multimodal=True
)

print('\n--- Baseline Model ---')
print(baseline_recs.to_string(index=False))

print('\n--- Multimodal Model ---')
print(multimodal_recs.to_string(index=False))

overlap = sorted(set(baseline_recs['movie_idx']).intersection(multimodal_recs['movie_idx']))
print(f'\nTop-10 overlap: {len(overlap)} movies -> {overlap}')

combined_recs = baseline_recs[['movie_idx', 'title', 'rank', 'predicted_rating']].rename(
    columns={'rank': 'baseline_rank', 'predicted_rating': 'baseline_score'}
).merge(
    multimodal_recs[['movie_idx', 'rank', 'predicted_rating']].rename(
        columns={'rank': 'multimodal_rank', 'predicted_rating': 'multimodal_score'}
    ),
    on='movie_idx',
    how='outer'
)
combined_recs['title'] = combined_recs['title'].fillna(
    combined_recs['movie_idx'].map(recommendation_catalog['final_title'])
)
combined_recs['score_delta'] = combined_recs['multimodal_score'] - combined_recs['baseline_score']
print('\n=== Recommendation Difference Table ===')
print(combined_recs.sort_values(['multimodal_rank', 'baseline_rank'], na_position='last').to_string(index=False))

plot_recs = pd.concat([
    baseline_recs.assign(model='Baseline NCF'),
    multimodal_recs.assign(model='Multimodal NCF'),
], ignore_index=True)
plot_recs['label'] = plot_recs['model'] + ' #' + plot_recs['rank'].astype(str)
plt.figure(figsize=(11, 6))
colors = plot_recs['model'].map({'Baseline NCF': 'steelblue', 'Multimodal NCF': 'coral'})
plt.barh(
    plot_recs['label'].iloc[::-1],
    plot_recs['predicted_rating'].iloc[::-1],
    color=colors.iloc[::-1].tolist(),
)
plt.xlim(RATING_MIN, RATING_MAX)
plt.xlabel('Predicted rating')
plt.title(f'Recommendation Scores for user_idx={DEMO_USER}')
plt.tight_layout()
plt.show()


## Save Models

The final trained models are saved under the project-level `saved_models/` folder so they are separate from the generated dataset files.


In [ ]:
save_dir = PROJECT_ROOT / 'saved_models'
save_dir.mkdir(parents=True, exist_ok=True)

baseline_model.save(save_dir / 'baseline_ncf.keras')
multi_model.save(save_dir / 'multimodal_ncf.keras')

print('Models saved to:', save_dir)
print('  baseline_ncf.keras')
print('  multimodal_ncf.keras')


## Full Dataset Note

`USE_SAMPLE = True` is best for fast experiments and presentations. Set `USE_SAMPLE = False` only when the machine has enough memory/time for the full MovieLens ratings file.
